In [2]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

env = gym.make('Blackjack-v1', natural=False, sab=False)

def mc_control_epsilon_greedy(env, num_episodes, epsilon=0.1, gamma=1.0):
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    returns = defaultdict(list)
    
    for i in range(num_episodes):
        if i % 10000 == 0:
            print(f"Episode {i}/{num_episodes}")
        episode = []
        state, _ = env.reset()
        done = False
        while not done:
            # Chọn hành động theo epsilon-greedy
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode.append((state, action, reward))
            state = next_state
        
        # First-visit MC: tính return và cập nhật Q
        visited = set()
        G = 0
        for t in range(len(episode)-1, -1, -1):
            state, action, reward = episode[t]
            G = gamma * G + reward
            if (state, action) not in visited:
                visited.add((state, action))
                returns[(state, action)].append(G)
                Q[state][action] = np.mean(returns[(state, action)])
    
    # Chính sách tối ưu: chọn hành động có Q cao nhất
    policy = {state: np.argmax(Q[state]) for state in Q}
    return policy, Q

# Chạy với 100.000 episode (giảm để nhanh hơn)
policy, Q = mc_control_epsilon_greedy(env, 100000, epsilon=0.1)

# In một vài trạng thái mẫu
sample_states = [(15, 6, False), (20, 6, False), (12, 2, True)]
for s in sample_states:
    action = policy.get(s, "chưa gặp")
    print(f"State {s}: hành động tốt nhất = {'Dừng' if action==0 else 'Rút'}")

Episode 0/100000
Episode 10000/100000
Episode 20000/100000
Episode 30000/100000
Episode 40000/100000
Episode 50000/100000
Episode 60000/100000
Episode 70000/100000
Episode 80000/100000
Episode 90000/100000
State (15, 6, False): hành động tốt nhất = Dừng
State (20, 6, False): hành động tốt nhất = Dừng
State (12, 2, True): hành động tốt nhất = Dừng
